In [21]:
# IMPORTS
# FIRST, pip install -r requirements.txt

import json
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
# PATHS
ROOT = Path(".")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results" / "playlists"

for d in (DATA_PROC, MODELS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

USER_FILES = {
    "andy": DATA_RAW / "enrichment-success-andy.json",
    "dishita": DATA_RAW / "enrichment-success-dish.json",
    "riya": DATA_RAW / "enrichment-success-riya.json",
    "priyanka": DATA_RAW / "enrichment-success-priyanka.json",
}

# TUNEABLE PARAMETERS FOR ENTIRE PIPELINE
AUDIO_FEATURES = [
    "danceability", "energy", "valence", "tempo", "loudness",
    "acousticness", "instrumentalness", "speechiness", "liveness",
    "key", "mode", "time_signature",
]
MOOD_FEATURES = ["valence", "energy", "danceability", "tempo"]
N_MOOD_CLUSTERS = 4
TOP_N_CONTENT = 300
TOP_N_TAG = 200
TOP_N_NEIGHBOR = 150
FINAL_CANDIDATE_K = 1000
TOP_K_TAGS = 10
MIN_NEIGHBOR_MS = 30_000
MAX_NEIGHBOR_SKIP = 0.3

In [23]:
# DATA LOADING AND PREPROCESSING

"""
Returns the best available tag list for a track
Tries all_tags first, falls back to sc_genres, then lastfm_tags
If none are available, returns an empty list
"""
def resolve_tags(row) -> list:
    """Canonical tag: all_tags → sc_genres → lastfm_tags."""
    for col in ("all_tags", "sc_genres", "lastfm_tags"):
        val = row.get(col)
        if isinstance(val, list) and len(val) > 0:
            return [t.strip().lower() for t in val]
    return []

"""
Loads a single user's raw (enriched) JSON listening history and returns it as a cleaned df
Filters out non-track plays (episodes, audiobooks), parses timestamps, and
adds flag for plays (>=30 seconds) that were not skipped.
"""
def load_user_events(path: Path, user: str) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    df = pd.json_normalize(raw)
    df['user'] = user

    af_rename = {f"audio_features.{feat}": feat for feat in AUDIO_FEATURES}
    df = df.rename(columns=af_rename)

    # Keep only track plays
    df = df[df["spotify_track_uri"].notna() & df["episode_name"].isna()].copy()

    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    df["ms_played"] = pd.to_numeric(df["ms_played"], errors="coerce").fillna(0)
    df["skipped"] = df["skipped"].fillna(False).astype(bool)
    df["tags"] = df.apply(resolve_tags, axis=1)

    # Meaningful play: not skipped AND listened to >= 30 seconds
    df["meaningful"] = (~df["skipped"]) & (df["ms_played"] >= 30_000)
    return df

"""
Loads and combines listening history for all users into a single df
Each row represents one play event, tagged with the user it belongs to
"""
def load_all_events(user_files: dict) -> pd.DataFrame:
    frames = [load_user_events(p, u) for u, p in user_files.items()]
    return pd.concat(frames, ignore_index=True)

events = load_all_events(USER_FILES)
events.to_csv(DATA_PROC / "events.csv", index=False)

print(f"Total play events: {len(events):,}")
print(f"Unique tracks: {events['spotify_track_uri'].nunique():,}")
print(f"Users: {events['user'].unique()}")

Total play events: 29,887
Unique tracks: 7,068
Users: <StringArray>
['andy', 'dishita', 'riya', 'priyanka']
Length: 4, dtype: str


In [24]:
# TRACK TABLE (ITEM VECTORS)

# TRACK TABLE
track_table = (
    events[[
        "spotify_track_uri",
        "master_metadata_track_name",
        "master_metadata_album_artist_name",
        "master_metadata_album_album_name",
        "tags",
        *AUDIO_FEATURES,
    ]]
    .drop_duplicates(subset="spotify_track_uri")
    .reset_index(drop=True)
    .rename(columns={
        "master_metadata_track_name": "track_name",
        "master_metadata_album_artist_name": "artist_name",
        "master_metadata_album_album_name": "album_name",
    })
)

track_table["track_id"] = track_table.index

uri_to_id = dict(zip(track_table["spotify_track_uri"], track_table["track_id"]))
id_to_uri = dict(zip(track_table["track_id"], track_table["spotify_track_uri"]))

# NORMALIZED AUDIO FEATURE MATRIX
audio_matrix = track_table[AUDIO_FEATURES].fillna(0).values.astype(float)
audio_scaler = MinMaxScaler()
audio_matrix_norm = audio_scaler.fit_transform(audio_matrix)

# TAG TF-IDF MATRIX
tag_docs = track_table["tags"].apply(lambda t: " ".join(t) if t else "")
tfidf = TfidfVectorizer(min_df=2, max_features=500)
tag_matrix = tfidf.fit_transform(tag_docs)
tag_vocab = tfidf.get_feature_names_out()

# ARTIST ID
track_table["artist_id"] = pd.Categorical(track_table["artist_name"]).codes
print(track_table.head())
print()

# SAVE TRACK TABLE & FITTED MODELS
track_table.to_csv(DATA_PROC / "track_features.csv", index=False)
joblib.dump(audio_scaler, MODELS_DIR / "audio_scaler.joblib")
joblib.dump(tfidf, MODELS_DIR / "tfidf.joblib")

print(f"Track table saved: {len(track_table):,} tracks, tag vocab = {len(tag_vocab)}")
print(f"Models saved: audio_scaler.joblib, tfidf.joblib → {MODELS_DIR}")
print()

                      spotify_track_uri                            track_name  \
0  spotify:track:1TycxSppuyhz7NVbiOs870  Triangle Ship (feat. Kendrick Lamar)   
1  spotify:track:59kp0ZIIrRNfILBzU8MnFD       Ab Soul's Intro (feat. Ab-Soul)   
2  spotify:track:5mDM8ebttXPLMkFxzrX8EA                              Get Away   
3  spotify:track:6k4ZH5yi5mIBXiFK8xJoGJ                         You Will Rise   
4  spotify:track:06aBnHJdZnuf8wZc6EhEHM                             トーキョー レギー   

          artist_name  album_name                                tags  \
0      Terrace Martin  3ChordFold            [hip hop, hip-hop & rap]   
1      Terrace Martin  3ChordFold            [hip hop, hip-hop & rap]   
2      Terrace Martin  3ChordFold            [hip hop, hip-hop & rap]   
3           Sweetback   Sweetback  [pop, r&b, r&b, funk & soul, rock]   
4  Masayoshi Takanaka   ALL OF ME            [jazz, pop, j-pop, rock]   

   danceability  energy  valence   tempo  loudness  acousticness  \
0     

In [25]:
# USER PROFILES



In [26]:
# SCORING

# RE-RANKING

# UI-WIDGETS (?)

# TESTING

# DEMO (?)